# GeoGebra applet interaction from Python

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
import xml.etree.ElementTree as ET
import xmlschema
import base64

In [3]:
from ggblab import GeoGebra

In [4]:
# open GeoGebra Widget on left-side
ggb = await GeoGebra().init()

Using local cached file: xsd/common.xsd


In [5]:
# case 1 of interactions: algebraic command
# [GeoGebra Manual :: GeoGebra Manual]
# (https://geogebra.github.io/docs/manual/)
r = await ggb.command("O = (0, 0)")
r

'O'

In [5]:
# case 2 of interactions: API function
# [GeoGebra Apps API :: GeoGebra Manual]
# (https://geogebra.github.io/docs/reference/en/GeoGebra_Apps_API/)
r = await ggb.function("getAllObjectNames")
r

['O']

In [6]:
r = await ggb.function("newConstruction")
r

In [7]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.file.load('2025_06_08.ggb')

In [8]:
# sending loaded construction to GeoGebra view and draw
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [9]:
from itertools import zip_longest

l0 = range(10)
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=False)))
l0 = [9, 0]
await ggb.function("setLayerVisible", list(zip_longest(list(l0), [], fillvalue=True)))

[None, None]

In [10]:
# case 3 of interactions: XML attributes

In [9]:
r = await ggb.function("getXML", ['f'])
print(r)

<element type="segment" label="f">
	<show object="true" label="true"/>
	<objColor r="0" g="0" b="0" alpha="0"/>
	<layer val="0"/>
	<labelMode val="0"/>
	<decoration type="4"/>
	<lineStyle thickness="5" type="10" typeHidden="1" opacity="178"/>
	<eqnStyle style="implicit"/>
	<outlyingIntersections val="false"/>
	<keepTypeOnTransform val="true"/>
	<startStyle val="arrow"/>
	<endStyle val="line"/>
	<coords x="0.16000000000000014" y="3.34" z="-5.3124"/>
</element>



In [12]:
o2 = c.ggb_schema.decode(r)
o2

{'@type': 'point',
 '@label': 'A',
 'show': [{'@object': True, '@label': True, '@ev': 4}],
 'objColor': [{'@r': 176, '@g': 0, '@b': 32, '@alpha': 0.0}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 0}],
 'animation': [{'@step': '0.1', '@type': 1, '@playing': False}],
 'pointSize': [{'@val': 5.0}],
 'pointStyle': [{'@val': 0}],
 'coords': [{'@x': '10', '@y': '0', '@z': '1'}]}

In [13]:
o2['show'][0]['@object'] = False
o2

{'@type': 'point',
 '@label': 'A',
 'show': [{'@object': False, '@label': True, '@ev': 4}],
 'objColor': [{'@r': 176, '@g': 0, '@b': 32, '@alpha': 0.0}],
 'layer': [{'@val': 9}],
 'labelMode': [{'@val': 0}],
 'animation': [{'@step': '0.1', '@type': 1, '@playing': False}],
 'pointSize': [{'@val': 5.0}],
 'pointStyle': [{'@val': 0}],
 'coords': [{'@x': '10', '@y': '0', '@z': '1'}]}

In [14]:
# construct xml attributes from dict
x = xmlschema.etree_tostring(c.ggb_schema.encode(o2, 'element'))
print(x)

<element type="point" label="A">
    <show object="false" label="true" ev="4" />
    <objColor r="176" g="0" b="32" alpha="0.0" />
    <layer val="9" />
    <labelMode val="0" />
    <animation step="0.1" type="1" playing="false" />
    <pointSize val="5.0" />
    <pointStyle val="0" />
    <coords x="10" y="0" z="1" />
</element>


In [15]:
# update the applet
r = await ggb.function("evalXML", [x])

In [16]:
r = await ggb.function("getBase64")

In [17]:
type(r), type(r.encode('ascii'))

(str, bytes)

In [18]:
# encode str to bytes for save
ggb.construction.base64_buffer = r.encode('ascii')
# ggb.construction.base64_buffer

In [19]:
# using API, some part of archive would lost...
c.save()